In [1]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'


Starting virtual X frame buffer: Xvfb../xvfb: line 24: start-stop-daemon: command not found
.


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [2]:
import numpy as np
import gymnasium as gym
from atari_wrappers import nature_dqn_env


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 8  # change this if you have more than 8 CPU ;)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]


Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device(
    'cuda' if torch.cuda.is_available() 
    else 'mps' if torch.mps.is_available() 
    else 'cpu'
    )

# those who have a GPU but feel unfair to use it can uncomment:
# device = torch.device('cpu')
# device
print(f"Using {device} device")


Using mps device


In [4]:
# import tensorflow as torch
# import torch as tf

class SmallNatureDQN(nn.Module):
    def __init__(self, n_actions, hidden_size=256):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(4, 16, 8, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, 4, stride=2),
            nn.ReLU(),

            # nn.Conv2d(16, 32, 4, stride=2),
            # nn.ReLU(),

            nn.Flatten(),
            # nn.Linear(32 * , 256),
            nn.LazyLinear(hidden_size),
            nn.ReLU()
        )

        self.action_head = nn.Linear(hidden_size, n_actions)
        self.value_head = nn.Linear(hidden_size, 1)

        def init_weights(layer):
            if isinstance(layer, nn.Conv2d):
                nn.init.orthogonal_(layer.weight)
                nn.init.zeros_(layer.bias)
        
        self.backbone.apply(init_weights)
            

    def forward(self, input: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        hidden = self.backbone(input)

        return self.action_head(hidden), self.value_head(hidden)


You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [5]:
from torch.distributions import Categorical

class Policy:
    def __init__(self, model):
        self.model = model

    # @torch.no_grad()
    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].
        inputs = torch.tensor(inputs, dtype=torch.float32, device=device)

        action_logits, values = self.model(inputs)

        dist = Categorical(logits=action_logits)
        sampled_actions = dist.sample()

        return {
            'actions': sampled_actions.detach().cpu().numpy(),
            'logits': action_logits,
            'log_probs': dist.log_prob(sampled_actions),
            'entropy': dist.entropy(),
            'values': values.squeeze(-1)
        }


Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [6]:
from runners import EnvRunner


This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [21]:
class ComputeValueTargets:
    def __init__(self, policy: Policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""
        value_target = self.policy.act(trajectory['state']['latest_observation'])["values"].detach().cpu().numpy()
        # value = act['values'].item()
        rewards = trajectory['rewards']

        value_targets = []
        for rewards, resets in zip(reversed(trajectory['rewards']), reversed(trajectory['resets'])):
            rewards = np.asarray(rewards, dtype=np.float32)
            resets = np.asarray(resets, dtype=np.float32)

            value_target = rewards + self.gamma * value_target * (1 - resets)
            value_targets.append(value_target)

        trajectory['value_targets'] = value_targets[::-1]


After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [25]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        # Modify trajectory inplace.

        for interaction in trajectory:
            if interaction == "state":
                continue 

            if torch.is_tensor(trajectory[interaction][0]):
                trajectory[interaction] = torch.stack(trajectory[interaction]).type(torch.float32).to(device).flatten(end_dim=1)
            else:
                trajectory[interaction] = torch.tensor(np.asarray(trajectory[interaction]), dtype=torch.float32, device=device).flatten(end_dim=1)


In [9]:
n_actions = env.action_space.n
state_dim = env.observation_space.shape


In [27]:
model = SmallNatureDQN(n_actions).to(device)
model(torch.zeros([1, 4, 84, 84], device=device))


(tensor([[-0.0087, -0.0296,  0.0138, -0.0473,  0.0225,  0.0188]],
        device='mps:0', grad_fn=<LinearBackward0>),
 tensor([[0.0302]], device='mps:0', grad_fn=<LinearBackward0>))

In [28]:

policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [12]:
class A2C:
    def __init__(self,
                 policy,
                 optimizer: torch.optim.Optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5,
                 ):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        # self.gamma = gamma

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        advantages = trajectory['value_targets'] - trajectory['values']

        return -torch.mean(trajectory['log_probs'] * advantages.detach())

    def value_loss(self, trajectory):
        value_diff = trajectory['values'] - trajectory['value_targets'].detach()
        return torch.mean(value_diff**2)

    def loss(self, trajectory):
        return (
            self.policy_loss(trajectory)
            + self.value_loss_coef * self.value_loss(trajectory)
            - self.entropy_coef * trajectory['entropy'].mean()
        )

    def step(self, trajectory):
        loss = self.loss(trajectory)
        trajectory['a2c_loss'] = loss
        self.optimizer.zero_grad()
        loss.backward()
        trajectory['grad_norms'] = nn.utils.clip_grad_norm_(self.policy.model.parameters(), self.max_grad_norm)
        self.optimizer.step()


Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [14]:
#if you use TensorboardSummaries
%load_ext tensorboard
%tensorboard --logdir logs


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 41660), started 0:00:15 ago. (Use '!kill 41660' to kill it.)

In [29]:
opt = torch.optim.Adam(model.parameters(), 1e-5)
a2c = A2C(policy, opt)

total_steps = 10_000_000

while runner.step_var < total_steps:
    trajectory = runner.get_next()
    a2c.step(trajectory)

    runner.add_summary('a2c_loss', trajectory['a2c_loss'].item())
    runner.add_summary('entropy', trajectory['entropy'].mean().item())
    runner.add_summary('value_targets', trajectory['value_targets'].mean().item())
    runner.add_summary('values', trajectory['values'].mean().item())
    runner.add_summary('grad_norm', trajectory['grad_norms'].item())


Process Process-5:
Process Process-6:
Process Process-8:
Process Process-7:
Process Process-1:
Process Process-2:
Process Process-3:
Process Process-4:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Users/timofejbulgakov/.pyenv/versions/3.11.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/timofejbulgakov/.pyenv/versions/3.11.11/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/timofejbulgakov/VSCodeProjects/Practical_RL/week06_policy_based/env_batch.py", line 148, in worker
    cmd, data = worker_connection.recv()
                ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/timofejbulgakov/.pyenv/versions/3.11.11/lib/python3.11/mu

KeyboardInterrupt: 

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.